In [1]:
#SIMULATION - MODEL (Readme)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Module conducts the following analysis:
# ---- Prepares offtaker dataset for analysis (aggregation of heavy duty, etc.)
# ---- Calculate predictor variables on the H2 offtaker dataset
# ---- Simulates green H2 demand baseline projections
# ---- Figure 5: Baseline projections of expected green H2 demand before policy intervention
# ---- Supplementary Figure 5: Baseline projections of expected green hydrogen demand before policy intervention using cogeneration plants as historical analogue 
#      (Uses Figure 5 code; produced when Empirics-Model is first run for cogeneration plants)
# ---- Supplementary Figure 7: Baseline projections of expected green hydrogen demand before policy intervention using wind plants as historical analogue 
#      (Uses Figure 5 code; produced when Empirics-Model is first run for wind plants)
# ---- Supplementary Figure 13: Baseline projections of expected green hydrogen demand before policy intervention using wind as historical analogue 
#      (Uses Figure 5 code; produced when Empirics-Model and Simulation-Model are first run for carbon_price_setting = flat_carbon_price)
# ---- Extended Data Tables 2 and 3
# ---- Extended Data Tables 2 and 3

# Module is input for:

# ---- Network-centrality (requires offtaker dataset)
# ---- Simulation-policy-intervention

In [2]:
#SET-UP
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
setwd("/home/h1604190/Spatially-informed Demand-side Policies for Green H2 Diffusion/") 
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.getenv("PROJ_LIB")

check_and_load <- function(packages) {
  for (pkg in packages) {
    if (!requireNamespace(pkg, quietly = TRUE)) {
      message(paste("Installing missing package:", pkg))
      install.packages(pkg, dependencies = TRUE, repos = "https://cloud.r-project.org")
    }
    if (!(pkg %in% (.packages()))) {
      suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    }
  }
}

# required libraries
required_packages <- c(
  # data handling
  "tidyverse",   # dplyr, ggplot2, tidyr, tibble, etc.
  "data.table",  # fast data manipulation
  "readxl",      # read Excel
  "writexl",     # write Excel
  "jsonlite",    # JSON  
  "purrr",       # data handling
  "openxlsx",    # export source data to XLSX
  
  # spatial analysis
  "sf",          # spatial data
  "giscoR",      # EU/NUTS geodata
  "Matrix",      # sparse matrices for spatial weights
  "FNN",         # nearest-neighbor spatial weights
  "geosphere",   # geographic calculations
  
  # visualization
  "ggsci",       # Nature/NPG palettes
  "patchwork",   # combine ggplots
  "cowplot",     # facet plot
  "ggplot2",     # Plotting

  # --- Simulation and forecasting utilities ---
  "forecast",    # Time-series and forecasting tools (trend extrapolation)
  "minpack.lm"   # Nonlinear least squares optimisation (Levenberg–Marquardt)
)

# --- Load all required packages (auto-install if missing) ------------
check_and_load(required_packages)


# Font
theme_set(
  theme_minimal(base_family = "Arial")
)

[1] "/opt/conda/share/proj"

In [3]:
#INPUTS AND SETTIGS
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Simulation
initial_share <- 0.01                       # Adopters at beginning of simulation initialized to 1% of total offtakers; note that calibration is 
carbon_price_setting <- "carbon_price"      # Insert carbon price assumption as "flat_carbon_price" or escalating "carbon_price"
rotterdam_coords <- cbind(4.4786, 51.9244)  # Rotterdam coordinates
distance_cutoff_rotterdam = 50000           # Distance cutoff for initialization
k_max_number <- 500                         # Max number of neighbors (FNN KNN used for efficient creation of spatial weights matrix)
km_cutoff = 120000                          # Distance cutoff for adjacency matrix (based on optimization loop, see above)
simulation_years    <- 2024:2100            # Simulation years
continuous_sectors  <- c("Heavy duty", "Aviation", "Shipping")  #Continuous instead of binary adoption

# simulation set-up
set.seed(123)
n_runs <- 250
base_seed <- 1000


#CRS
crs <- 3035                                 # EPSG:3035 - LAEA Europe

# Saturation rates - reflecting competition vs other technologies (details in Supplementary Note 6)
define_saturation <- function(version = c("restricted", "central", "extended")) {

  version <- match.arg(version)

  sectors <- c("Aviation",
               "Chemicals",
               "Heat",
               "Heavy duty",
               "Iron & steel",
               "Non-ferrous metals",
               "Non-metallic minerals",
               "Other",
               "Power",
               "Pulp & paper",
               "Refining",
               "Shipping")

  values <- switch(version,

    restricted = c(
      0.00,  # Aviation
      0.35,  # Chemicals
      0.00,  # Heat
      0.00,  # Heavy duty
      0.20,  # Iron & steel
      0.00,  # Non-ferrous metals
      0.00,  # Non-metallic minerals
      0.00,  # Other
      0.00,  # Power
      0.00,  # Pulp & paper
      0.45,  # Refining
      0.00   # Shipping
    ),

    central = c(
      0.35,  # Aviation
      0.70,  # Chemicals
      0.00,  # Heat
      0.25,  # Heavy duty
      0.40,  # Iron & steel
      0.25,  # Non-ferrous metals
      0.20,  # Non-metallic minerals
      0.05,  # Other
      0.10,  # Power
      0.00,  # Pulp & paper
      0.60,  # Refining
      0.35   # Shipping
    ),

    extended = c(
      0.80,  # Aviation
      0.90,  # Chemicals
      0.05,  # Heat
      0.35,  # Heavy duty
      0.70,  # Iron & steel
      0.50,  # Non-ferrous metals
      0.70,  # Non-metallic minerals
      0.10,  # Other
      1.00,  # Power
      0.10,  # Pulp & paper
      0.75,  # Refining
      0.60   # Shipping
    )
  )

  setNames(values, sectors)
}

In [4]:
#DATA FILES
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
#Paths
inet_pipesegments_geojson_path <- "Data/INET_PipeSegments.geojson" 
h2_database_path <- "Data/h2_database.xlsx"

#Load datafiles
pipe_segments <- st_read(inet_pipesegments_geojson_path) 
grid_clean <-  suppressWarnings(read_excel(h2_database_path))
grid_clean_sf <- grid_clean %>%
  st_drop_geometry() %>%
  mutate(
    lon = as.numeric(lon),
    lat = as.numeric(lat)
  ) %>%
  st_as_sf(coords = c("lon", "lat"), crs = 4326, remove = FALSE)
maritime_sf <- grid_clean_sf %>% filter(Industry == "Shipping") #filter maritime port subset
iww_sf <- grid_clean_sf %>% filter(Industry == "Inland Shipping") #filter inland port subset

# Load NUTS shapefiles
nuts2_shapefile <- gisco_get_nuts(year = 2021, nuts_level = 2, resolution = "20")
if (is.na(st_crs(nuts2_shapefile))) st_crs(nuts2_shapefile) <- 4258
nuts2_shapefile <- st_transform(nuts2_shapefile, crs = 4326)

Reading layer `INET_PipeSegments' from data source 
  `/home/h1604190/Spatially-informed Demand-side Policies for Green H2 Diffusion/Data/INET_PipeSegments.geojson' 
  using driver `GeoJSON'
Simple feature collection with 920 features and 6 fields
Geometry type: LINESTRING
Dimension:     XY
Bounding box:  xmin: -9.788 ymin: 36.12883 xmax: 28.76455 ymax: 61.66
Geodetic CRS:  WGS 84


In [5]:
# Load data from empirical model - comment out if no access to S&P Capital IQ data
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
second_pass_coefs <- readRDS("second_pass_coefs.rds") #comment out if there is no access to the data underlying the empirical module
cost_gap_data <- readRDS("cost_gap_data.rds") #comment out if there is no access to the data underlying the empirical module

In [6]:
'#SPECIAL INPUTS - use in case there is no access to the S&P Capital IQ data underlying the empirical model
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
#note that this chunk is relevant only in case there is no access to the underlying data for the empirics module. This chunk allows to run the simulation based on the
#results of the empirical analysis. Comment out if you are able to run the empirical analysis. 

# Datafiles
second_pass_coefs <- "Data/second_pass_coefs.rds"
cost_green_path <- "Data/cost_competitiveness_green.xlsx"
cost_fossil_path <- "Data/cost_competitiveness_fossil.xlsx"

cost_gap_green <- read_excel(cost_green_path) %>%
  pivot_longer(`2024`:`2050`, names_to = "year", values_to = "green_cost") %>%
  mutate(year = as.integer(year))
cost_gap_fossil <- read_excel(cost_fossil_path) %>%
  pivot_longer(`2024`:`2050`, names_to = "year", values_to = "fossil_cost") %>%
  mutate(year = as.integer(year))

# Cost forecast settings
future_years <- 2051:2100                   # Cost competitiveness forecast extension years
currency <- "EUR"                           # Set to EUR or USD
exchange_rate <- 1/1.0321                   # ECB as of 2 Jan 25
# H2 sector mapping
sector_mapping <- tribble(
  ~sector,                ~green_commodity,       ~fossil_commodity, 
  "Aviation",             "E-Kerosine",           "Kerosine", 
  "Chemicals",            "Green Hydrogen",       "Grey Hydrogen", 
  "Heat",                 "E-Methane",            "Natural Gas", 
  "Heavy duty",           "Green Hydrogen Mobility", "Diesel", 
  "Iron & steel",         "Green Hydrogen Steel", "Natural Gas", 
  "Non-ferrous metals",   "Green Hydrogen",       "Natural Gas",
  "Non-metallic minerals","Green Hydrogen",       "Natural Gas",
  "Other",                "Green Hydrogen",       "Natural Gas",
  "Power",                "E-Methane",            "Natural Gas",
  "Pulp & paper",         "Green Hydrogen",       "Natural Gas", 
  "Refining",             "Green Hydrogen",       "Grey Hydrogen",
  "Shipping",             "E-Methanol",           "Diesel"
)

# Forecast functions
wrights_law_forecast <- function(years, values, future_years) {
  C_2050 <- values[years == 2050]
  anchored <- function(t, r) C_2050 * exp(-r * (t - 2050))
  fit <- nlsLM(values ~ anchored(years, r),
               start = list(r = 0.05), lower = 0.001, upper = 1,
               control = nls.lm.control(maxiter = 500))
  r_est <- coef(fit)[["r"]]
  C_2050 * exp(-r_est * (future_years - 2050))
}

arima_forecast <- function(years, values, future_years) {
  if (length(unique(values)) < 5) return(rep(NA, length(future_years)))
  ts_data <- ts(values, start = min(years), frequency = 1)
  forecast(auto.arima(ts_data), h = length(future_years))$mean |> as.numeric()
}

ar1_forecast <- function(years, values, future_years) {
  if (length(unique(values)) < 5) return(rep(NA, length(future_years)))

  ts_data <- ts(values, start = min(years), frequency = 1)

  fit <- arima(
    ts_data,
    order = c(1, 0, 0),
    method = "ML",
    transform.pars = TRUE   # forces |phi| < 1
  )

  preds <- predict(fit, n.ahead = length(future_years))$pred
  as.numeric(preds)
}


# Forecast data
forecast_green <- cost_gap_green %>%
  group_by(commodity, green_scenario) %>%
  group_map(~{
    if (nrow(.x) < 5) return(NULL)
    tibble(
      commodity = .y$commodity,
      green_scenario = .y$green_scenario,
      year = future_years,
      green_cost = wrights_law_forecast(.x$year, .x$green_cost, future_years)
    )
  }) %>% bind_rows()

forecast_fossil <- cost_gap_fossil %>%
  group_by(commodity, fossil_scenario) %>%
  group_map(~{

    if (nrow(.x) < 5) return(NULL)

    is_flat <- (.y$fossil_scenario == "flat_carbon_price")

    tibble(
      commodity       = .y$commodity,
      fossil_scenario = .y$fossil_scenario,
      year            = future_years,
      fossil_cost     = if (is_flat) {
        # AR(1) for flat carbon price
        ar1_forecast(.x$year, .x$fossil_cost, future_years)
      } else {
        # ARIMA for all default carbon price
        arima_forecast(.x$year, .x$fossil_cost, future_years)
      }
    )
  }) %>%
  bind_rows()

# Combine data
green_all  <- bind_rows(cost_gap_green,  forecast_green)
fossil_all <- bind_rows(cost_gap_fossil, forecast_fossil)
scenario_grid <- expand.grid(
  green_scenario = unique(green_all$green_scenario),
  fossil_scenario = unique(fossil_all$fossil_scenario),
  year = sort(unique(c(green_all$year, fossil_all$year)))
)

# Map to sector
sector_data <- sector_mapping %>%
  crossing(scenario_grid) %>%
  left_join(green_all,  by = c("green_commodity" = "commodity", "green_scenario", "year")) %>%
  left_join(fossil_all, by = c("fossil_commodity" = "commodity", "fossil_scenario", "year")) %>%
  filter(!is.na(green_cost), !is.na(fossil_cost)) %>%
  mutate(
    across(c(green_cost, fossil_cost),
           ~ if (currency == "EUR") .x / exchange_rate else .x),
    cost_diff = fossil_cost - green_cost,
    scenario_pair = paste0("green: ", green_scenario, " | fossil: ", fossil_scenario)
  ) %>%
  select(
    sector, year, green_commodity, fossil_commodity,
    green_scenario, fossil_scenario, scenario_pair,
    green_cost, fossil_cost, cost_diff
  )

#Extract cost competitiveness values
cost_gap_data <- sector_data %>%
  filter(fossil_scenario == carbon_price_setting) #Based on carbon price setting in global settings
table(cost_gap_data$sector) #Check -> should be 231 for all sectors
saveRDS(cost_gap_data, "cost_gap_data.rds") #Save for use in simulation modules'

[1] "#SPECIAL INPUTS - use in case there is no access to the S&P Capital IQ data underlying the empirical model\n#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------\n#note that this chunk is relevant only in case there is no access to the underlying data for the empirics module. This chunk allows to run the simulation based on the\n#results of the empirical analysis. Comment out if you are able to run the empirical analysis. \n\n# Datafiles\nsecond_pass_coefs <- \"Data/second_pass_coefs.rds\"\ncost_green_path <- \"Data/cost_competitiveness_green.xlsx\"\ncost_fossil_path <- \"Data/cost_competitiveness_fossil.xlsx\"\n\ncost_gap_green <- read_excel(cost_green_path) %>%\n  pivot_longer(`2024`:`2050`, names_to = \"year\", values_to = \"green_cost\") %>%\n  mutate(year = as.integer(year))\ncost_gap_fossil <- read_excel(cost_fossil_path) %>%\n  pivot_longer(`2024`:`2050`, names_to = \"year\", values_to = \"fossil_cost\") %>%\n  mutate(year = as.integer(year))\n\n# Cost forecast settings\nfuture_years <- 2051:2100                   # Cost competitiveness forecast extension years\ncurrency <- \"EUR\"                           # Set to EUR or USD\nexchange_rate <- 1/1.0321                   # ECB as of 2 Jan 25\n# H2 sector mapping\nsector_mapping <- tribble(\n  ~sector,                ~green_commodity,       ~fossil_commodity, \n  \"Aviation\",             \"E-Kerosine\",           \"Kerosine\", \n  \"Chemicals\",            \"Green Hydrogen\",       \"Grey Hydrogen\", \n  \"Heat\",                 \"E-Methane\",            \"Natural Gas\", \n  \"Heavy duty\",           \"Green Hydrogen Mobility\", \"Diesel\", \n  \"Iron & steel\",         \"Green Hydrogen Steel\", \"Natural Gas\", \n  \"Non-ferrous metals\",   \"Green Hydrogen\",       \"Natural Gas\",\n  \"Non-metallic minerals\",\"Green Hydrogen\",       \"Natural Gas\",\n  \"Other\",                \"Green Hydrogen\",       \"Natural Gas\",\n  \"Power\",                \"E-Methane\",            \"Natural Gas\",\n  \"Pulp & paper\",         \"Green Hydrogen\",       \"Natural Gas\", \n  \"Refining\",             \"Green Hydrogen\",       \"Grey Hydrogen\",\n  \"Shipping\",             \"E-Methanol\",           \"Diesel\"\n)\n\n# Forecast functions\nwrights_law_forecast <- function(years, values, future_years) {\n  C_2050 <- values[years == 2050]\n  anchored <- function(t, r) C_2050 * exp(-r * (t - 2050))\n  fit <- nlsLM(values ~ anchored(years, r),\n               start = list(r = 0.05), lower = 0.001, upper = 1,\n               control = nls.lm.control(maxiter = 500))\n  r_est <- coef(fit)[[\"r\"]]\n  C_2050 * exp(-r_est * (future_years - 2050))\n}\n\narima_forecast <- function(years, values, future_years) {\n  if (length(unique(values)) < 5) return(rep(NA, length(future_years)))\n  ts_data <- ts(values, start = min(years), frequency = 1)\n  forecast(auto.arima(ts_data), h = length(future_years))$mean |> as.numeric()\n}\n\nar1_forecast <- function(years, values, future_years) {\n  if (length(unique(values)) < 5) return(rep(NA, length(future_years)))\n\n  ts_data <- ts(values, start = min(years), frequency = 1)\n\n  fit <- arima(\n    ts_data,\n    order = c(1, 0, 0),\n    method = \"ML\",\n    transform.pars = TRUE   # forces |phi| < 1\n  )\n\n  preds <- predict(fit, n.ahead = length(future_years))$pred\n  as.numeric(preds)\n}\n\n\n# Forecast data\nforecast_green <- cost_gap_green %>%\n  group_by(commodity, green_scenario) %>%\n  group_map(~{\n    if (nrow(.x) < 5) return(NULL)\n    tibble(\n      commodity = .y$commodity,\n      green_scenario = .y$green_scenario,\n      year = future_years,\n      green_cost = wrights_law_forecast(.x$year, .x$green_cost, future_years)\n    )\n  }) %>% bind_rows()\n\nforecast_fossil <- cost_gap_fossil %>%\n  group_by(commodity, fossil_scenario) %>%\n  group_map(~{\n\n    if (nrow(.x) < 5) r

In [7]:
#DATA HANDLING
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# expand json fields in pipeline segment data
pipe_segments_expanded <- pipe_segments %>%
  mutate(
    param  = map(param,  ~ fromJSON(.) %>% as.data.frame()),
    method = map(method, ~ fromJSON(.) %>% as.data.frame())
  ) %>%
  unnest_wider(param,  names_sep = "_") %>%
  unnest_wider(method, names_sep = "_")

# convert offtaker table to sf and keep europe-relevant extent
grid_clean_sf <- st_as_sf(
  as.data.frame(grid_clean),
  coords = c("lon", "lat"),
  crs = 4326
) %>%
  filter(
    st_coordinates(.)[, 1] >= -30 & st_coordinates(.)[, 1] <= 40,
    st_coordinates(.)[, 2] >= 30  & st_coordinates(.)[, 2] <= 72
  ) %>%
  mutate(index = row_number())

# extract coordinates again and calculate distance to rotterdam
grid_clean_sf <- grid_clean_sf %>%
  mutate(
    coords = st_coordinates(geometry),
    lon = coords[, 1],
    lat = coords[, 2],
    distance_to_rotterdam = distHaversine(cbind(lon, lat), rotterdam_coords) / 1000
  ) %>%
  select(-coords, -geometry)

# aggregate heavy-duty demand to nuts3 level
grid_clean_hd <- grid_clean_sf %>%
  filter(Sector == "Heavy duty")

grid_clean_hd_aggregated <- grid_clean_hd %>%
  group_by(contact_country, NUTS3_code) %>%
  summarise(
    hydrogen_sum = sum(hydrogen, na.rm = TRUE),
    hydrogen_sum_2030 = sum(hydrogen_2030, na.rm = TRUE),
    hydrogen_sum_2050 = sum(hydrogen_2050, na.rm = TRUE),
    mean_distance_to_rotterdam = mean(distance_to_rotterdam, na.rm = TRUE),
    geometry = st_union(geometry),
    .groups = "drop"
  ) %>%
  mutate(centroid = st_centroid(geometry))

# assign representative original point to each aggregated heavy-duty region
nearest_points <- st_nearest_feature(grid_clean_hd_aggregated$centroid, grid_clean_hd)

representative_points <- grid_clean_hd[nearest_points, c("NUTS3_code", "geometry")]

grid_clean_hd_aggregated <- grid_clean_hd_aggregated %>%
  left_join(
    representative_points %>%
      st_drop_geometry() %>%
      mutate(
        lon = st_coordinates(grid_clean_hd[nearest_points, ])[, 1],
        lat = st_coordinates(grid_clean_hd[nearest_points, ])[, 2]
      ),
    by = "NUTS3_code"
  ) %>%
  select(-centroid) %>%
  rename(
    distance_to_rotterdam = mean_distance_to_rotterdam
  ) %>%
  mutate(
    hydrogen = hydrogen_sum,
    hydrogen_2030 = hydrogen_sum_2030,
    hydrogen_2050 = hydrogen_sum_2050,
    Sector = "Heavy duty",
    Industry = "Heavy duty trucks"
  )

# recombine aggregated heavy-duty with all other sectors
grid_clean_abm <- grid_clean_sf %>%
  filter(Sector != "Heavy duty") %>%
  bind_rows(grid_clean_hd_aggregated) %>%
  select(-geometry, -index)

grid_clean_abm_sf <- st_as_sf(grid_clean_abm, coords = c("lon", "lat"), crs = 4326)

# build final simulation dataset
simulation_data <- grid_clean_abm_sf %>%
  mutate(offtaker_id = row_number()) %>%
  rename(sector = Sector) %>%
  st_drop_geometry() %>%
  filter(!is.na(lat)) %>%
  st_as_sf(coords = c("lon", "lat"), crs = 4326) %>%
  st_transform(crs = crs)

# transform infrastructure layers to common crs
maritime_sf      <- st_transform(maritime_sf, crs)
iww_sf           <- st_transform(iww_sf, crs)
pipe_segments_sf <- st_transform(pipe_segments, crs)

# calculate minimum distances to infrastructure
distance_to_port     <- apply(st_distance(simulation_data, maritime_sf), 1, min) / 1000
distance_to_iww      <- apply(st_distance(simulation_data, iww_sf), 1, min) / 1000
distance_to_pipeline <- apply(st_distance(simulation_data, pipe_segments_sf), 1, min) / 1000

# store distance fields in simulation data
simulation_data <- simulation_data %>%
  mutate(
    plant_id = row_number(),
    distance_to_rotterdam = distance_to_rotterdam,
    distance_to_port = distance_to_port,
    distance_to_iww = distance_to_iww,
    distance_to_pipeline = distance_to_pipeline,
    distance_to_waterway = pmin(distance_to_port, distance_to_iww, na.rm = TRUE)
  )

In [8]:
#EXTENDED DATA TABLES 2 and 3
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# aggregation by country
table_country <- simulation_data %>%
  group_by(contact_country) %>%
  summarise(
    n = n(),  # number of plants
    total_h2_2050 = sum(hydrogen_2050, na.rm = TRUE),  # total demand
    avg_h2_2050 = mean(hydrogen_2050, na.rm = TRUE)    # average per plant
  ) %>%
  arrange(desc(n))

# aggregation by sector
table_sector <- simulation_data %>%
  group_by(sector) %>%
  summarise(
    n = n(),
    total_h2_2050 = sum(hydrogen_2050, na.rm = TRUE),
    avg_h2_2050 = mean(hydrogen_2050, na.rm = TRUE)
  ) %>%
  arrange(desc(n))

# aggregation by industry (not shown in paper)
table_industry <- simulation_data %>%
  group_by(Industry) %>%
  summarise(
    n = n(),
    total_h2_2050 = sum(hydrogen_2050, na.rm = TRUE),
    avg_h2_2050 = mean(hydrogen_2050, na.rm = TRUE)
  ) %>%
  arrange(desc(n))

# print full tables
print(table_country, n = Inf)
print(table_sector, n = Inf)
print(table_industry, n = Inf)

Simple feature collection with 27 features and 4 fields
Geometry type: MULTIPOINT
Dimension:     XY
Bounding box:  xmin: 1093639 ymin: 1430662 xmax: 6493040 ymax: 5143791
Projected CRS: ETRS89-extended / LAEA Europe
# A tibble: 27 × 5
   contact_country     n total_h2_2050 avg_h2_2050                      geometry
   <chr>           <int>         <dbl>       <dbl>              <MULTIPOINT [m]>
 1 DE               3365        41485.       12.3  ((4040181 3105590), (4042125…
 2 FR               1788        27640.       15.5  ((3245848 2914581), (3255586…
 3 IT               1723        22489.       13.1  ((4060481 2446008), (4101156…
 4 ES               1380        18178.       13.2  ((2769730 2405353), (2775048…
 5 BE                881        10209.       11.6  ((3803273 3133093), (3806478…
 6 PL                776        11426.       14.7  ((4599751 3322548), (4599968…
 7 NL                598        15991.       26.7  ((3874038 3162540), (3878198…
 8 RO                394         346

In [9]:
offtakers <- simulation_data %>%
  select(
    plant_id, installation_name, account_holder_name, sector,
    distance_to_rotterdam, distance_to_port, distance_to_iww,
    distance_to_pipeline, distance_to_waterway,
    hydrogen, hydrogen_2050, emissions, contact_country
  ) %>%
  mutate(
    adoption = 0,
    previous_adoption = 0,
    p = 0
  )

get_coef <- function(name) {
  if (name %in% names(second_pass_coefs)) as.numeric(second_pass_coefs[[name]]) else 0
}

beta0_scalar <- get_coef("(Intercept)")
beta1_scalar <- get_coef("spatial_influence_detrended")
beta2_scalar <- get_coef("cost_proxy_scaled")
beta3_scalar <- get_coef("distance_to_waterway")
beta4_scalar <- get_coef("distance_to_pipeline")
beta5_scalar <- get_coef("cost_proxy_scaled:spatial_influence_detrended")

offtakers <- offtakers %>%
  mutate(
    beta0 = beta0_scalar,
    beta1 = beta1_scalar,
    beta2 = beta2_scalar,
    beta3 = beta3_scalar,
    beta4 = beta4_scalar,
    beta5 = beta5_scalar
  )

adopt_prob <- function(beta0, beta1, spatial_influence_detrended,
                       beta2, cost_diff,
                       beta3, distance_to_waterway,
                       beta4, distance_to_pipeline,
                       beta5) {

  1 / (1 + exp(-(beta0 +
                 beta1 * spatial_influence_detrended +
                 beta2 * cost_diff +
                 beta3 * distance_to_waterway +
                 beta4 * distance_to_pipeline +
                 beta5 * spatial_influence_detrended * cost_diff)))
}

offtakers <- offtakers %>%
  mutate(
    beta6_restricted = recode(sector, !!!define_saturation("restricted"), .default = 0),
    beta6_central    = recode(sector, !!!define_saturation("central"), .default = 0),
    beta6_extended   = recode(sector, !!!define_saturation("extended"), .default = 0)
  )

compute_distance_weights <- function(offtakers, distance_cutoff = km_cutoff, k_max = k_max_number) {
  coords <- st_coordinates(offtakers)

  knn <- get.knnx(data = coords, query = coords, k = k_max)

  i_vec <- rep(seq_len(nrow(coords)), each = k_max)
  j_vec <- as.vector(knn$nn.index)
  d_vec <- as.vector(knn$nn.dist)

  valid <- which(d_vec > 0 & d_vec <= distance_cutoff)

  i <- i_vec[valid]
  j <- j_vec[valid]

  W <- sparseMatrix(i = i, j = j, x = 1, dims = c(nrow(coords), nrow(coords)))
  W_norm <- W / pmax(rowSums(W), 1)

  list(W = W_norm, neighbors_matrix = W)
}

weights <- compute_distance_weights(offtakers)
spatial_weights  <- weights$W
neighbors_matrix <- weights$neighbors_matrix

scenarios <- expand.grid(
  saturation = c("restricted", "central", "extended"),
  green_scenario = c("conservative", "progressive", "mean"),
  stringsAsFactors = FALSE
)

continuous_sectors <- c("Heavy duty", "Aviation", "Shipping")

annual_results <- vector("list", length = nrow(scenarios) * n_runs)
sector_results <- vector("list", length = nrow(scenarios) * n_runs)
result_counter <- 1

for (i in seq_len(nrow(scenarios))) {

  sat <- scenarios$saturation[i]
  gs  <- scenarios$green_scenario[i]
  label <- paste(sat, "|", gs)

  message("Running scenario: ", label)

  offtakers_base <- offtakers %>%
    mutate(
      beta6 = get(paste0("beta6_", sat))
    )

  cost_diff_lookup <- cost_gap_data %>%
    filter(
      green_scenario == gs,
      fossil_scenario == carbon_price_setting
    ) %>%
    select(year, sector, cost_diff) %>%
    split(.$year)

  for (run_id in seq_len(n_runs)) {

    set.seed(base_seed + i * 10000 + run_id)

    offtakers_run <- offtakers_base

    eligible_init <- offtakers_run %>%
      mutate(row_id = row_number()) %>%
      filter(
        distance_to_rotterdam <= distance_cutoff_rotterdam,
        beta6 > 0
      )

    if (nrow(eligible_init) > 0) {
      selected_indices <- sample(
        eligible_init$row_id,
        size = min(nrow(eligible_init), max(1, round(initial_share * nrow(eligible_init)))),
        replace = FALSE
      )
    } else {
      selected_indices <- integer(0)
    }

    offtakers_run <- offtakers_run %>%
      mutate(
        previous_adoption = if_else(row_number() %in% selected_indices, 1, 0)
      )

    cumulative_adoption <- offtakers_run$previous_adoption

    annual_run <- vector("list", length(simulation_years))
    sector_run <- vector("list", sum(simulation_years %in% c(2030, 2050)))
    sector_counter <- 1

    for (t in seq_along(simulation_years)) {

      current_year <- simulation_years[t]

      spatial_influence <- as.numeric(spatial_weights %*% cumulative_adoption)
      spatial_influence[is.na(spatial_influence)] <- 0
      spatial_detrended <- spatial_influence - mean(spatial_influence)

      cost_year <- cost_diff_lookup[[as.character(current_year)]]

      if (is.null(cost_year)) {
        cost_year <- tibble(sector = unique(offtakers_run$sector), cost_diff = 0)
      } else {
        cost_year <- distinct(cost_year, sector, .keep_all = TRUE)
      }

      cost_diff <- offtakers_run %>%
        select(sector) %>%
        left_join(cost_year, by = "sector") %>%
        mutate(cost_diff = replace_na(cost_diff, 0)) %>%
        pull(cost_diff)

      p_raw <- adopt_prob(
        offtakers_run$beta0,
        offtakers_run$beta1, spatial_detrended,
        offtakers_run$beta2, cost_diff,
        offtakers_run$beta3, offtakers_run$distance_to_waterway,
        offtakers_run$beta4, offtakers_run$distance_to_pipeline,
        offtakers_run$beta5
      )
      p_raw[is.na(p_raw)] <- 0

      sector_summary <- tibble(
        sector = offtakers_run$sector,
        cumulative_adoption = cumulative_adoption,
        beta6 = offtakers_run$beta6
      ) %>%
        group_by(sector) %>%
        summarise(
          sector_mean = mean(cumulative_adoption),
          beta6 = mean(beta6),
          .groups = "drop"
        ) %>%
        mutate(
          residual_share = pmax(0, (beta6 - sector_mean) / pmax(1e-6, 1 - sector_mean))
        )

      residual_p <- tibble(sector = offtakers_run$sector, p = p_raw) %>%
        left_join(sector_summary, by = "sector") %>%
        transmute(residual_p = replace_na(p, 0) * replace_na(residual_share, 0)) %>%
        pull(residual_p)

      residual_p[is.na(residual_p)] <- 0

      for (s in unique(offtakers_run$sector)) {
        idx_s <- which(offtakers_run$sector == s)

        if (s %in% continuous_sectors) {
          cumulative_adoption[idx_s] <-
            1 - (1 - cumulative_adoption[idx_s]) * (1 - residual_p[idx_s])
        } else {
          cumulative_adoption[idx_s] <- pmax(
            cumulative_adoption[idx_s],
            rbinom(length(idx_s), 1, residual_p[idx_s])
          )
        }
      }

      cumulative_adoption[offtakers_run$beta6 == 0] <- 0

      h2_uptake <- cumulative_adoption * offtakers_run$hydrogen_2050

      annual_run[[t]] <- tibble(
        run = run_id,
        year = current_year,
        scenario = sat,
        green_scenario = gs,
        h2_uptake_mt = sum(h2_uptake, na.rm = TRUE) / 1000
      )

      if (current_year %in% c(2030, 2050)) {
        sector_run[[sector_counter]] <- tibble(
          year = current_year,
          sector = offtakers_run$sector,
          h2_uptake = h2_uptake
        ) %>%
          group_by(year, sector) %>%
          summarise(
            h2_uptake = sum(h2_uptake, na.rm = TRUE) / 1000,
            .groups = "drop"
          ) %>%
          mutate(
            run = run_id,
            scenario = sat,
            green_scenario = gs
          )

        sector_counter <- sector_counter + 1
      }
    }

    annual_results[[result_counter]] <- bind_rows(annual_run)
    sector_results[[result_counter]] <- bind_rows(sector_run)
    result_counter <- result_counter + 1
  }
}

h2_summary_runs <- bind_rows(annual_results)
sector_bars_runs <- bind_rows(sector_results)

h2_summary <- h2_summary_runs %>%
  group_by(year, scenario, green_scenario) %>%
  summarise(
    avg_h2_uptake = mean(h2_uptake_mt, na.rm = TRUE),
    sd_h2_uptake = sd(h2_uptake_mt, na.rm = TRUE),
    p05_h2_uptake = quantile(h2_uptake_mt, 0.05, na.rm = TRUE),
    p50_h2_uptake = quantile(h2_uptake_mt, 0.50, na.rm = TRUE),
    p95_h2_uptake = quantile(h2_uptake_mt, 0.95, na.rm = TRUE),
    .groups = "drop"
  )

sector_bars <- sector_bars_runs %>%
  group_by(year, sector, green_scenario, scenario) %>%
  summarise(
    h2_uptake = mean(h2_uptake, na.rm = TRUE),
    .groups = "drop"
  )

saveRDS(h2_summary_runs, "h2_summary_runs.rds")
saveRDS(h2_summary, "h2_summary.rds")
saveRDS(sector_bars_runs, "sector_bars_runs.rds")
saveRDS(sector_bars, "sector_bars.rds")
saveRDS(offtakers, "offtakers.rds")
saveRDS(neighbors_matrix, "neighbors_matrix.rds")
saveRDS(spatial_weights, "spatial_weights.rds")

Running scenario: restricted | conservative

Running scenario: central | conservative

Running scenario: extended | conservative

Running scenario: restricted | progressive

Running scenario: central | progressive

Running scenario: extended | progressive

Running scenario: central | mean

Running scenario: extended | mean



In [ ]:
# FIGURE 5: H2 DEMAND 2030, 2050, DIFFUSION
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

grey_palette <- c(
  "conservative" = "grey20",
  "mean"         = "grey50",
  "progressive"  = "grey80"
)

all_sectors <- sort(c(
  "Power", "Heavy duty", "Pulp & paper", "Chemicals", "Refining",
  "Iron & steel", "Non-ferrous metals", "Other", "Aviation",
  "Shipping", "Non-metallic minerals", "Heat"
))

sector_palette <- setNames(
  colorRampPalette(pal_npg("nrc")(10))(length(all_sectors)),
  all_sectors
)

tmp_iron <- sector_palette["Iron & steel"]
tmp_nf   <- sector_palette["Non-ferrous metals"]
tmp_heat <- sector_palette["Heat"]

sector_palette["Iron & steel"]       <- tmp_nf
sector_palette["Heat"]               <- tmp_iron
sector_palette["Non-ferrous metals"] <- tmp_heat

plot_theme <- theme_minimal(base_size = 18) +
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.position = "right",
    legend.box = "vertical",
    legend.title = element_text(face = "plain", size = 18),
    legend.text = element_text(size = 18),
    legend.key.width = unit(1.2, "cm"),
    legend.key.height = unit(0.5, "cm"),
    panel.spacing = unit(1.2, "lines"),
    panel.border = element_rect(color = "black", fill = NA),
    axis.line = element_line(color = "black"),
    axis.ticks = element_blank(),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 18),
    axis.text.y = element_text(size = 18),
    axis.text.x.top = element_blank(),
    axis.text.y.right = element_blank(),
    axis.title.x = element_text(size = 18),
    axis.title.y = element_blank(),
    strip.text = element_text(size = 18),
    plot.title = element_text(hjust = 0.5, size = 18)
  )

h2_summary_plot <- h2_summary %>%
  mutate(
    scenario = factor(scenario, levels = c("restricted", "central", "extended")),
    green_scenario = factor(green_scenario, levels = c("conservative", "mean", "progressive"))
  )

sector_bars_plot <- sector_bars %>%
  filter(year %in% c(2030, 2050)) %>%
  mutate(
    scenario = factor(scenario, levels = c("restricted", "central", "extended")),
    green_scenario = factor(green_scenario, levels = c("conservative", "mean", "progressive")),
    sector = factor(sector, levels = all_sectors)
  )



bar_ref_lines <- tibble(
  year = c(2030, 2050),
  yintercept = c(20, 50),
  label = "Policy Target"
)

line_ref_lines <- tidyr::crossing(
  scenario = factor(
    c("restricted", "central", "extended"),
    levels = c("restricted", "central", "extended")
  ),
  tibble(
    yintercept = c(20, 50),
    target_year = c(2030, 2050),
    label = c("2030 Target", "2050 Target")
  )
)

line_plot <- ggplot(
  h2_summary_plot,
  aes(x = year, y = avg_h2_uptake, color = green_scenario)
) +
  geom_line(linewidth = 1.2) +
  geom_hline(
    data = line_ref_lines,
    aes(yintercept = yintercept, linetype = label),
    inherit.aes = FALSE,
    color = "black",
    linewidth = 0.6
  ) +
  geom_text(
    data = line_ref_lines,
    aes(x = 2073, y = yintercept, label = label),
    inherit.aes = FALSE,
    hjust = 0,
    vjust = -0.4,
    size = 5,
    color = "black"
  ) +
  scale_color_manual(
    values = grey_palette,
    name = "Cost Competitiveness",
    labels = c(
      conservative = "Conservative",
      mean         = "Mean",
      progressive  = "Progressive"
    )
  ) +
  scale_linetype_manual(
    values = c("2030 Target" = "dashed", "2050 Target" = "dashed"),
    name = ""
  ) +
  labs(
    x = "Year",
    title = "Diffusion over Time"
  ) +
  facet_grid(
    rows = vars(scenario),
    scales = "fixed",
    labeller = labeller(
      scenario = c(
        restricted = "Restricted",
        central    = "Central",
        extended   = "Extended"
      )
    )
  ) +
  coord_cartesian(clip = "off") +
  plot_theme +
  theme(
    strip.text.y = element_blank(),
    axis.text.y  = element_blank(),
    axis.ticks.y = element_blank(),
    axis.title.y = element_blank()
  )

bar_matrix <- ggplot(
  sector_bars_plot,
  aes(x = green_scenario, y = h2_uptake, fill = sector)
) +
  geom_col(width = 0.7) +
  geom_hline(
    data = bar_ref_lines,
    aes(yintercept = yintercept, linetype = label),
    inherit.aes = FALSE,
    color = "black",
    linewidth = 0.6
  ) +
  facet_grid(
    rows = vars(scenario),
    cols = vars(year),
    switch = "y",
    labeller = labeller(
      scenario = c(
        restricted = "Restricted",
        central    = "Central",
        extended   = "Extended"
      )
    )
  ) +
  scale_fill_manual(
    values = sector_palette,
    name = "Sector",
    drop = FALSE
  ) +
  scale_x_discrete(
    labels = c(
      conservative = "Conservative",
      mean         = "Mean",
      progressive  = "Progressive"
    )
  ) +
  scale_linetype_manual(
    values = c("Policy Target" = "dashed"),
    name = ""
  ) +
  labs(
    x = "Cost Competitiveness",
    y = "Green H₂ Demand (Mt)",
    title = "Expected Green H₂ Demand by Sector"
  ) +
  coord_cartesian(clip = "off") +
  plot_theme

combined <- (bar_matrix | line_plot) +
  plot_layout(guides = "collect", widths = c(8, 6)) +
  plot_annotation(tag_levels = "a") &
  theme(
    plot.tag = element_text(size = 18),
    legend.position = "right",
    axis.title.y = element_blank()
  )

figure5 <- ggdraw(combined) +
  draw_label(
    "Green H₂ Demand (Mt)",
    x = 0,
    y = 0.5,
    vjust = 1.5,
    angle = 90,
    size = 18
  )

options(repr.plot.width = 16, repr.plot.height = 10, repr.plot.res = 600)
print(figure5)

ggsave(
  "figure5.pdf",
  figure5,
  device = cairo_pdf,
  width = 16,
  height = 14,
  units = "in",
  dpi = 800
)

In [ ]:
# EXPORT FIGURE 5 SOURCE DATA TO XLSX
# -------------------------------------------------------------------

# Folder / filename
out_file <- "figure5_source_data.xlsx"


# Figure metadata

figure_metadata <- tibble(
  item = c(
    "Figure",
    "Description",
    "Demand unit",
    "Time horizon",
    "Scenarios",
    "Cost competitiveness scenarios",
    "Sector categories",
    "Policy targets"
  ),
  value = c(
    "Figure 5: Green H2 demand 2030, 2050, diffusion",
    "Expected green hydrogen demand by sector and diffusion trajectory",
    "Mt H2",
    "2024-2070",
    "Restricted, Central, Extended",
    "Conservative, Mean, Progressive",
    paste(all_sectors, collapse = ", "),
    "20 Mt in 2030; 50 Mt in 2050"
  )
)


# Source sheets

source_sheets <- list(

  "README" = figure_metadata,


  # Panel a: sector demand bars
  "Panel_a_sector_demand" = sector_bars %>%
    filter(year %in% c(2030, 2050)) %>%
    mutate(
      scenario = as.character(scenario),
      green_scenario = as.character(green_scenario)
    ) %>%
    select(
      scenario,
      year,
      green_scenario,
      sector,
      h2_uptake
    ),


  # Panel b: diffusion trajectories
  "Panel_b_diffusion" = h2_summary %>%
    mutate(
      scenario = as.character(scenario),
      green_scenario = as.character(green_scenario)
    ) %>%
    select(
      scenario,
      year,
      green_scenario,
      avg_h2_uptake
    ),


  # Policy reference lines
  "Policy_targets" = tibble(
    target_year = c(2030, 2050),
    target_demand_Mt = c(20, 50),
    label = c(
      "2030 Target",
      "2050 Target"
    )
  ),


  # Scenario definitions
  "Scenario_definitions" = tibble(
    scenario = c(
      "restricted",
      "central",
      "extended"
    ),
    description = c(
      "Restricted adoption assumptions",
      "Central adoption assumptions",
      "Extended adoption assumptions"
    )
  ),


  # Cost assumptions
  "Cost_scenarios" = tibble(
    green_scenario = c(
      "conservative",
      "mean",
      "progressive"
    ),
    description = c(
      "Higher green hydrogen costs",
      "Mean cost assumptions",
      "Lower green hydrogen costs"
    )
  )
)


# Write workbook

wb <- createWorkbook()

walk(names(source_sheets), function(sheet_name) {

  addWorksheet(
    wb,
    sheet_name
  )

  writeData(
    wb,
    sheet = sheet_name,
    x = source_sheets[[sheet_name]]
  )

  freezePane(
    wb,
    sheet = sheet_name,
    firstRow = TRUE
  )

  addFilter(
    wb,
    sheet = sheet_name,
    row = 1,
    cols = seq_len(ncol(source_sheets[[sheet_name]]))
  )

  setColWidths(
    wb,
    sheet = sheet_name,
    cols = seq_len(ncol(source_sheets[[sheet_name]])),
    widths = "auto"
  )
})


saveWorkbook(
  wb,
  out_file,
  overwrite = TRUE
)

In [ ]:
# TOP 5 SECTORS BY ACTUAL ADOPTED DEMAND
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

sector_bars_from_runs <- sector_bars_runs %>%
  group_by(year, sector, green_scenario, scenario) %>%
  summarise(
    h2_uptake = mean(h2_uptake, na.rm = TRUE),
    .groups = "drop"
  )

top5_sector_shares <- sector_bars_from_runs %>%
  filter(year %in% c(2050)) %>%
  group_by(year, scenario, green_scenario) %>%
  mutate(
    total_adopted_demand = sum(h2_uptake, na.rm = TRUE),
    share = h2_uptake / total_adopted_demand
  ) %>%
  arrange(year, scenario, green_scenario, desc(h2_uptake)) %>%
  group_by(year, scenario, green_scenario) %>%
  slice_head(n = 5) %>%
  mutate(
    rank = row_number(),
    share_pct = share * 100,
    adopted_demand = h2_uptake
  ) %>%
  ungroup() %>%
  select(
    year, scenario, green_scenario, rank, sector, share_pct, share, adopted_demand
  ) %>%
  arrange(year, scenario, green_scenario, rank)

print(top5_sector_shares, n = Inf, width = Inf)

# A tibble: 45 × 8
    year scenario   green_scenario  rank sector                share_pct  share
   <int> <chr>      <chr>          <int> <chr>                     <dbl>  <dbl>
 1  2050 central    conservative       1 Chemicals                 81.5  0.815 
 2  2050 central    conservative       2 Iron & steel               8.62 0.0862
 3  2050 central    conservative       3 Aviation                   2.95 0.0295
 4  2050 central    conservative       4 Heavy duty                 2.45 0.0245
 5  2050 central    conservative       5 Power                      1.18 0.0118
 6  2050 central    mean               1 Chemicals                 62.2  0.622 
 7  2050 central    mean               2 Iron & steel              13.8  0.138 
 8  2050 central    mean               3 Aviation                   7.11 0.0711
 9  2050 central    mean               4 Non-metallic minerals      4.92 0.0492
10  2050 central    mean               5 Heavy duty                 4.65 0.0465
11  2050 central    p

In [ ]:
# DEMAND VERSUS POLICY TARGETS
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

h2_summary_from_runs <- h2_summary_runs %>%
  group_by(year, scenario, green_scenario) %>%
  summarise(
    avg_h2_uptake = mean(h2_uptake_mt, na.rm = TRUE),
    .groups = "drop"
  )

h2_2030_2050_wide <- h2_summary_from_runs %>%
  filter(year %in% c(2030, 2050)) %>%
  select(year, scenario, green_scenario, avg_h2_uptake) %>%
  tidyr::pivot_wider(
    names_from = year,
    values_from = avg_h2_uptake,
    names_prefix = "demand_"
  ) %>%
  mutate(
    shortfall_2030 = pmax(0, 20 - demand_2030),
    shortfall_2050 = pmax(0, 50 - demand_2050),
    pct_shortfall_2030 = pmax(0, (20 - demand_2030) / 20 * 100),
    pct_shortfall_2050 = pmax(0, (50 - demand_2050) / 50 * 100)
  ) %>%
  arrange(scenario, green_scenario)

print(h2_2030_2050_wide, n = Inf, width = Inf)

# A tibble: 9 × 8
  scenario   green_scenario demand_2030 demand_2050 shortfall_2030
  <chr>      <chr>                <dbl>       <dbl>          <dbl>
1 central    conservative         0.928        9.74          19.1 
2 central    mean                 1.68        25.3           18.3 
3 central    progressive         10.2         38.4            9.80
4 extended   conservative         1.98        14.0           18.0 
5 extended   mean                 3.00        44.1           17.0 
6 extended   progressive         15.8         81.9            4.21
7 restricted conservative         0.336        4.54          19.7 
8 restricted mean                 0.590        9.91          19.4 
9 restricted progressive          3.71         9.83          16.3 
  shortfall_2050 pct_shortfall_2030 pct_shortfall_2050
           <dbl>              <dbl>              <dbl>
1          40.3                95.4               80.5
2          24.7                91.6               49.5
3          11.6          

In [ ]:
# solve backwards for required h2 cost to reach 2030 targets

target_year <- 2030
target_h2_mt <- 20
n_calibration_runs <- n_runs
calibration_green_scenario <- "mean"

target_tolerance_mt <- 0.10
cost_search_tolerance_eur_mwh <- 0.5

offtakers <- simulation_data %>%
  select(
    plant_id, installation_name, account_holder_name, sector,
    distance_to_rotterdam, distance_to_port, distance_to_iww,
    distance_to_pipeline, distance_to_waterway,
    hydrogen, hydrogen_2050, emissions, contact_country
  ) %>%
  mutate(
    adoption = 0,
    previous_adoption = 0,
    p = 0
  )

get_coef <- function(name) {
  if (name %in% names(second_pass_coefs)) as.numeric(second_pass_coefs[[name]]) else 0
}

beta0_scalar <- get_coef("(Intercept)")
beta1_scalar <- get_coef("spatial_influence_detrended")
beta2_scalar <- get_coef("cost_proxy_scaled")
beta3_scalar <- get_coef("distance_to_waterway")
beta4_scalar <- get_coef("distance_to_pipeline")
beta5_scalar <- get_coef("cost_proxy_scaled:spatial_influence_detrended")

offtakers <- offtakers %>%
  mutate(
    beta0 = beta0_scalar,
    beta1 = beta1_scalar,
    beta2 = beta2_scalar,
    beta3 = beta3_scalar,
    beta4 = beta4_scalar,
    beta5 = beta5_scalar
  )

adopt_prob <- function(beta0, beta1, spatial_influence_detrended,
                       beta2, cost_diff,
                       beta3, distance_to_waterway,
                       beta4, distance_to_pipeline,
                       beta5) {

  1 / (1 + exp(-(beta0 +
                   beta1 * spatial_influence_detrended +
                   beta2 * cost_diff +
                   beta3 * distance_to_waterway +
                   beta4 * distance_to_pipeline +
                   beta5 * spatial_influence_detrended * cost_diff)))
}

offtakers <- offtakers %>%
  mutate(
    beta6_restricted = recode(sector, !!!define_saturation("restricted"), .default = 0),
    beta6_central    = recode(sector, !!!define_saturation("central"), .default = 0),
    beta6_extended   = recode(sector, !!!define_saturation("extended"), .default = 0)
  )

compute_distance_weights <- function(offtakers, distance_cutoff = km_cutoff, k_max = k_max_number) {

  coords <- st_coordinates(offtakers)

  knn <- get.knnx(data = coords, query = coords, k = k_max)

  i_vec <- rep(seq_len(nrow(coords)), each = k_max)
  j_vec <- as.vector(knn$nn.index)
  d_vec <- as.vector(knn$nn.dist)

  valid <- which(d_vec > 0 & d_vec <= distance_cutoff)

  i <- i_vec[valid]
  j <- j_vec[valid]

  W <- sparseMatrix(i = i, j = j, x = 1, dims = c(nrow(coords), nrow(coords)))
  W_norm <- W / pmax(rowSums(W), 1)

  list(W = W_norm, neighbors_matrix = W)
}

weights <- compute_distance_weights(offtakers)

spatial_weights  <- weights$W
neighbors_matrix <- weights$neighbors_matrix

original_scenarios <- expand.grid(
  saturation = c("restricted", "central", "extended"),
  green_scenario = c("conservative", "progressive", "mean"),
  stringsAsFactors = FALSE
) %>%
  mutate(
    scenario_index = row_number()
  )

scenarios <- original_scenarios %>%
  filter(green_scenario == calibration_green_scenario)

continuous_sectors <- c("Heavy duty", "Aviation", "Shipping")

cost_gap_calibration_data <- cost_gap_data %>%
  filter(
    year <= target_year,
    green_scenario == calibration_green_scenario,
    fossil_scenario == carbon_price_setting
  )

green_h2_cost_trajectory <- cost_gap_data %>%
  filter(
    year <= target_year,
    green_scenario == calibration_green_scenario,
    green_commodity == "Green Hydrogen"
  ) %>%
  distinct(year, green_cost) %>%
  arrange(year)

green_cost_reference_2030 <- green_h2_cost_trajectory %>%
  filter(year == target_year) %>%
  distinct(green_cost) %>%
  pull(green_cost)

if (length(green_cost_reference_2030) != 1 || is.na(green_cost_reference_2030)) {
  stop("No unique 2030 direct Green Hydrogen cost found for the selected calibration scenario.")
}

cost_gap_calibration_data <- cost_gap_calibration_data %>%
  mutate(
    baseline_green_hydrogen_cost_eur_mwh = green_cost_reference_2030
  )


max_saturation_check <- offtakers %>%
  st_drop_geometry() %>%
  mutate(
    max_restricted = beta6_restricted * hydrogen_2050,
    max_central    = beta6_central    * hydrogen_2050,
    max_extended   = beta6_extended   * hydrogen_2050
  ) %>%
  summarise(
    max_restricted_mt = sum(max_restricted, na.rm = TRUE) / 1000,
    max_central_mt    = sum(max_central,    na.rm = TRUE) / 1000,
    max_extended_mt   = sum(max_extended,   na.rm = TRUE) / 1000
  )

print(max_saturation_check)

cost_gap_check <- green_h2_cost_trajectory %>%
  summarise(
    calibration_green_scenario = calibration_green_scenario,
    fossil_scenario = carbon_price_setting,
    n_years = n(),
    min_green_h2_cost = min(green_cost, na.rm = TRUE),
    mean_green_h2_cost = mean(green_cost, na.rm = TRUE),
    max_green_h2_cost = max(green_cost, na.rm = TRUE),
    baseline_green_hydrogen_cost_eur_mwh = green_cost_reference_2030
  )

print(cost_gap_check)

simulate_2030_uptake <- function(sat, scenario_index, green_hydrogen_cost_2030,
                                 n_runs_use = n_calibration_runs) {

  beta6_col <- paste0("beta6_", sat)

  if (!beta6_col %in% names(offtakers)) {
    stop("Column not found in offtakers: ", beta6_col)
  }

  cost_scale <- green_hydrogen_cost_2030 / green_cost_reference_2030

  offtakers_base <- offtakers %>%
    mutate(
      beta6 = .data[[beta6_col]]
    )

  cost_diff_lookup <- cost_gap_calibration_data %>%
    mutate(
      calibrated_green_cost = green_cost * cost_scale,
      calibrated_cost_diff  = fossil_cost - calibrated_green_cost
    ) %>%
    select(year, sector, calibrated_cost_diff) %>%
    split(.$year)

  uptake_runs <- numeric(n_runs_use)

  for (run_id in seq_len(n_runs_use)) {

    set.seed(base_seed + scenario_index * 10000 + run_id)

    offtakers_run <- offtakers_base

    eligible_init <- offtakers_run %>%
      mutate(row_id = row_number()) %>%
      filter(
        distance_to_rotterdam <= distance_cutoff_rotterdam,
        beta6 > 0
      )

    if (nrow(eligible_init) > 0) {
      selected_indices <- sample(
        eligible_init$row_id,
        size = min(nrow(eligible_init), max(1, round(initial_share * nrow(eligible_init)))),
        replace = FALSE
      )
    } else {
      selected_indices <- integer(0)
    }

    offtakers_run <- offtakers_run %>%
      mutate(
        previous_adoption = if_else(row_number() %in% selected_indices, 1, 0)
      )

    cumulative_adoption <- offtakers_run$previous_adoption

    for (current_year in simulation_years[simulation_years <= target_year]) {

      spatial_influence <- as.numeric(spatial_weights %*% cumulative_adoption)
      spatial_influence[is.na(spatial_influence)] <- 0

      spatial_detrended <- spatial_influence - mean(spatial_influence)

      cost_year <- cost_diff_lookup[[as.character(current_year)]]

      if (is.null(cost_year)) {
        cost_year <- tibble(
          sector = unique(offtakers_run$sector),
          calibrated_cost_diff = 0
        )
      } else {
        cost_year <- distinct(cost_year, sector, .keep_all = TRUE)
      }

      cost_diff <- offtakers_run %>%
        select(sector) %>%
        left_join(cost_year, by = "sector") %>%
        mutate(
          cost_diff = replace_na(calibrated_cost_diff, 0)
        ) %>%
        pull(cost_diff)

      p_raw <- adopt_prob(
        offtakers_run$beta0,
        offtakers_run$beta1, spatial_detrended,
        offtakers_run$beta2, cost_diff,
        offtakers_run$beta3, offtakers_run$distance_to_waterway,
        offtakers_run$beta4, offtakers_run$distance_to_pipeline,
        offtakers_run$beta5
      )

      p_raw[is.na(p_raw)] <- 0

      sector_summary <- tibble(
        sector = offtakers_run$sector,
        cumulative_adoption = cumulative_adoption,
        beta6 = offtakers_run$beta6
      ) %>%
        group_by(sector) %>%
        summarise(
          sector_mean = mean(cumulative_adoption),
          beta6 = mean(beta6),
          .groups = "drop"
        ) %>%
        mutate(
          residual_share = pmax(0, (beta6 - sector_mean) / pmax(1e-6, 1 - sector_mean))
        )

      residual_p <- tibble(
        sector = offtakers_run$sector,
        p = p_raw
      ) %>%
        left_join(sector_summary, by = "sector") %>%
        transmute(
          residual_p = replace_na(p, 0) * replace_na(residual_share, 0)
        ) %>%
        pull(residual_p)

      residual_p[is.na(residual_p)] <- 0

      for (s in unique(offtakers_run$sector)) {

        idx_s <- which(offtakers_run$sector == s)

        if (s %in% continuous_sectors) {
          cumulative_adoption[idx_s] <-
            1 - (1 - cumulative_adoption[idx_s]) * (1 - residual_p[idx_s])
        } else {
          cumulative_adoption[idx_s] <- pmax(
            cumulative_adoption[idx_s],
            rbinom(length(idx_s), 1, residual_p[idx_s])
          )
        }
      }

      cumulative_adoption[offtakers_run$beta6 == 0] <- 0
    }

    uptake_runs[run_id] <- sum(
      cumulative_adoption * offtakers_run$hydrogen_2050,
      na.rm = TRUE
    ) / 1000
  }

  mean(uptake_runs, na.rm = TRUE)
}

find_required_green_hydrogen_cost <- function(sat, scenario_index, lower = 0, upper = 1000) {

  baseline <- simulate_2030_uptake(
    sat = sat,
    scenario_index = scenario_index,
    green_hydrogen_cost_2030 = green_cost_reference_2030
  )

  lower_uptake <- simulate_2030_uptake(
    sat = sat,
    scenario_index = scenario_index,
    green_hydrogen_cost_2030 = lower
  )

  upper_uptake <- simulate_2030_uptake(
    sat = sat,
    scenario_index = scenario_index,
    green_hydrogen_cost_2030 = upper
  )

  if (lower_uptake < target_h2_mt - target_tolerance_mt) {
    return(
      tibble(
        saturation = sat,
        calibration_green_scenario = calibration_green_scenario,
        fossil_scenario = carbon_price_setting,
        scenario_index = scenario_index,
        baseline_green_hydrogen_cost_eur_mwh = green_cost_reference_2030,
        required_green_hydrogen_cost_eur_mwh = NA_real_,
        cost_scale = NA_real_,
        required_cost_change_eur_mwh = NA_real_,
        baseline_2030_mt = baseline,
        calibrated_2030_mt = lower_uptake,
        gap_to_target_mt = lower_uptake - target_h2_mt,
        target_reached = FALSE,
        reason = "target infeasible even at zero green H2 cost"
      )
    )
  }

  if (baseline >= target_h2_mt - target_tolerance_mt) {
    return(
      tibble(
        saturation = sat,
        calibration_green_scenario = calibration_green_scenario,
        fossil_scenario = carbon_price_setting,
        scenario_index = scenario_index,
        baseline_green_hydrogen_cost_eur_mwh = green_cost_reference_2030,
        required_green_hydrogen_cost_eur_mwh = green_cost_reference_2030,
        cost_scale = 1,
        required_cost_change_eur_mwh = 0,
        baseline_2030_mt = baseline,
        calibrated_2030_mt = baseline,
        gap_to_target_mt = baseline - target_h2_mt,
        target_reached = abs(baseline - target_h2_mt) <= target_tolerance_mt,
        reason = "target already met at baseline cost"
      )
    )
  }

  if (upper_uptake > target_h2_mt + target_tolerance_mt) {
    warning(
      "Target still exceeded at upper cost bound (", upper, ") for sat=", sat,
      ". Consider increasing `upper`."
    )

    return(
      tibble(
        saturation = sat,
        calibration_green_scenario = calibration_green_scenario,
        fossil_scenario = carbon_price_setting,
        scenario_index = scenario_index,
        baseline_green_hydrogen_cost_eur_mwh = green_cost_reference_2030,
        required_green_hydrogen_cost_eur_mwh = NA_real_,
        cost_scale = NA_real_,
        required_cost_change_eur_mwh = NA_real_,
        baseline_2030_mt = baseline,
        calibrated_2030_mt = upper_uptake,
        gap_to_target_mt = upper_uptake - target_h2_mt,
        target_reached = FALSE,
        reason = "target still exceeded at upper cost bound; increase `upper`"
      )
    )
  }

  root <- uniroot(
    function(x) {
      simulate_2030_uptake(
        sat = sat,
        scenario_index = scenario_index,
        green_hydrogen_cost_2030 = x
      ) - target_h2_mt
    },
    lower = lower,
    upper = upper,
    tol = cost_search_tolerance_eur_mwh
  )

  calibrated <- simulate_2030_uptake(
    sat = sat,
    scenario_index = scenario_index,
    green_hydrogen_cost_2030 = root$root
  )

  gap_to_target <- calibrated - target_h2_mt

  tibble(
    saturation = sat,
    calibration_green_scenario = calibration_green_scenario,
    fossil_scenario = carbon_price_setting,
    scenario_index = scenario_index,
    baseline_green_hydrogen_cost_eur_mwh = green_cost_reference_2030,
    required_green_hydrogen_cost_eur_mwh = root$root,
    cost_scale = root$root / green_cost_reference_2030,
    required_cost_change_eur_mwh = root$root - green_cost_reference_2030,
    baseline_2030_mt = baseline,
    calibrated_2030_mt = calibrated,
    gap_to_target_mt = gap_to_target,
    target_reached = abs(gap_to_target) <= target_tolerance_mt,
    reason = if_else(
      abs(gap_to_target) <= target_tolerance_mt,
      "target reached within tolerance",
      "root estimate outside target tolerance; check stochasticity"
    )
  )
}

required_green_hydrogen_costs <- purrr::pmap_dfr(
  list(
    sat = scenarios$saturation,
    scenario_index = scenarios$scenario_index
  ),
  find_required_green_hydrogen_cost
)

required_green_hydrogen_costs %>%
  filter(
    saturation == "central",
    calibration_green_scenario == "mean"
  ) %>%
  select(
    saturation,
    calibration_green_scenario,
    fossil_scenario,
    baseline_green_hydrogen_cost_eur_mwh,
    required_green_hydrogen_cost_eur_mwh,
    cost_scale,
    required_cost_change_eur_mwh,
    baseline_2030_mt,
    calibrated_2030_mt,
    gap_to_target_mt,
    target_reached,
    reason
  ) %>%
  print(width = Inf)

  max_restricted_mt max_central_mt max_extended_mt
1          9.537081       38.39779         82.1485
# A tibble: 1 × 7
  calibration_green_scenario fossil_scenario n_years min_green_h2_cost
  <chr>                      <chr>             <int>             <dbl>
1 mean                       carbon_price          7              129.
# ℹ 3 more variables: mean_green_h2_cost <dbl>, max_green_h2_cost <dbl>,
#   baseline_green_hydrogen_cost_eur_mwh <dbl>
